# Exercise : Vectorized pairwise distance & periodic boundary conditions

## Problem Statement

Calculating pairwise distances between $N$ particles in $D$-dimensional space is a core computational bottleneck in molecular dynamics simulations, atomic graph neural networks (GNNs), and physical system modeling.

When simulating physical systems inside a bounded box, we often enforce **Periodic Boundary Conditions (PBC)** to eliminate surface boundary effects. Under PBC, space wraps around: if a particle exits the box on the right, it re-enters from the left. Consequently, the relative distance between any two particles must be computed using the **minimum image convention**—the shortest vector across periodic boundaries.

Your task is to write a vectorized tensor function that computes the full pairwise distance matrix and extracts edge connections below a specified cutoff radius.

---

## Mathematical Background

1. **Unbounded pairwise displacement:**
Given particle coordinates $X \in \mathbb{R}^{N \times D}$, the pairwise vector from particle $j$ to particle $i$ is:

$$\Delta X_{i, j} = X_i - X_j \in \mathbb{R}^{D}$$


2. **Minimum Image Convention (Periodic Correction):**
For a cubic box with dimensions $L \in \mathbb{R}^D$, if $\vert{}\Delta X_{i, j, d}\vert{} > \frac{L_d}{2}$, the vector crosses a periodic boundary. The corrected displacement is:

$$\Delta X_{i, j, d}^{\text{pbc}} = \Delta X_{i, j, d} - L_d \cdot \text{round}\left(\frac{\Delta X_{i, j, d}}{L_d}\right)$$


3. **Pairwise Euclidean distance:**

$$D_{i, j} = \Vert{}\Delta X_{i, j}^{\text{pbc}}\Vert{}_2 = \sqrt{\sum_{d=1}^D \left(\Delta X_{i, j, d}^{\text{pbc}}\right)^2}$$



---

## Requirements

* Vectorization: your implementation must not contain any Python `for` or `while` loops. Use array broadcasting and array ops.
* Batching/dimensions: handle inputs of shape $(N, D)$ and compute intermediate tensors of shape $(N, N, D)$ via broadcasting.
* Adjacency extraction: return both the full $(N, N)$ distance matrix and a $(2, E)$ matrix of sparse edge indices $(i, j)$ where $D_{i, j} < \text{edge\_threshold}$.

In [1]:
from typing import Any

import numpy as np

NDArray = np.ndarray[Any, Any]

In [ ]:
def compute_pairwise_distances_pbc(positions: NDArray, box_size: NDArray) -> NDArray:
    """
    Computes pairwise relative distances between particles under periodic boundary conditions.

    Args:
        positions: Tensor of shape (N, D) containing particle coordinates.
        box_size: Tensor of shape (D,) specifying box dimensions along each axis.

    Returns:
        distances: Tensor of shape (N, N) containing pairwise Euclidean distances.
    """
    # 1. Unbounded relative displacement matrix via broadcasting: (N, 1, D) - (1, N, D) -> (N, N, D)

    # 2. Minimum image convention (PBC correction):
    # Subtract integer box multiples so displacements fall within [-box/2, box/2]

    # 3. Euclidean norm across coordinate axis D -> shape (N, N)



## Verification: unit testing

In ndarray computations, bugs don't always trigger explicit syntax or runtime errors. Instead, they can manifest as silent mathematical bugs caused by incorrect broadcasting dimensions, improper reduction axes, or unhandled geometric edge cases.

Write tests to ensure the following properties:

1. **Tensor shape & invariants:**
   * Dimensions: For an input positions tensor of shape `(N, D)`, the output matrix must strictly be of shape `(N, N)`.
   * Symmetry: The distance matrix must be symmetric: $D_{i, j} = D_{j, i}$ for all pairs $(i, j)$.
   * Zero Diagonal: The distance from any particle to itself must be zero ($D_{i, i} = 0$).
   * Non-Negativity: All computed distances must be non-negative ($D_{i, j} \ge 0$).

2. **Periodic Boundary Edge Case (Minimum Image Convention):**
   * Wraparound test: Consider a simple box of size $L = 10.0$ with two particles:
     * Particle 1 at coordinate $x = 0.5$
     * Particle 2 at coordinate $x = 9.5$
   * Expected Result: A naive distance calculation yields $9.0$. However, wrapping across the periodic boundary yields a true relative distance of **$1.0$**. Verify that your code computes wrapped distances correctly across all axes.

Use `assert` and [np.testing.assert_allclose](https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_allclose.html).  

In [ ]:
def test_pbc_wrapping_correctness():
    """Tests distance calculation with periodic boundary wrapping across axes."""



def test_distance_matrix_properties():
    """Tests general mathematical invariants: symmetry and zero self-distance."""


test_pbc_wrapping_correctness()
test_distance_matrix_properties()